In [ ]:
import re

def preprocess_stt_text(text):
    """
    STT 결과를 분석용으로 최소 전처리한다.
    의미를 바꾸는 수정은 하지 않는다.
    """

    if text is None:
        return ""

    text = str(text)

    # 앞뒤 공백 제거
    text = text.strip()

    # 라벨/비언어 표기 제거
    # 예: (NO:), (웃음), [잡음], <침묵>
    text = re.sub(r"\((NO:|웃음|기침|잡음|소음|침묵|무음|음악)\)", " ", text)
    text = re.sub(r"\[(NO:|웃음|기침|잡음|소음|침묵|무음|음악)\]", " ", text)
    text = re.sub(r"<(NO:|웃음|기침|잡음|소음|침묵|무음|음악)>", " ", text)

    # NO:가 괄호 없이 붙는 경우
    text = re.sub(r"\bNO:\s*", " ", text)

    # 줄바꿈, 탭, 여러 공백 정리
    text = re.sub(r"\s+", " ", text)

    # 양끝 따옴표 제거
    text = text.strip(" \"'“”‘’")

    return text.strip()

In [ ]:
samples = [
    "남자들은 그거 싸먹는 거 귀찮아하는데",
    "약간의 차이가 있네요 넷플릭스는 잘 안 봐요.",
    "(NO:)차차차 뭐 이런 거 있잖아요 언니?",
    "  [잡음] 좋은 운동 방법 있으면은 저도 좀 부탁할게요.  ",
    "그러니까 주식하고 잃을 여유가 없는 거지.",
]

for s in samples:
    print("원본:", s)
    print("전처리:", preprocess_stt_text(s))
    print("-" * 50)

원본: 남자들은 그거 싸먹는 거 귀찮아하는데
전처리: 남자들은 그거 싸먹는 거 귀찮아하는데
--------------------------------------------------
원본: 약간의 차이가 있네요 넷플릭스는 잘 안 봐요.
전처리: 약간의 차이가 있네요 넷플릭스는 잘 안 봐요.
--------------------------------------------------
원본: (NO:)차차차 뭐 이런 거 있잖아요 언니?
전처리: 차차차 뭐 이런 거 있잖아요 언니?
--------------------------------------------------
원본:   [잡음] 좋은 운동 방법 있으면은 저도 좀 부탁할게요.  
전처리: 좋은 운동 방법 있으면은 저도 좀 부탁할게요.
--------------------------------------------------
원본: 그러니까 주식하고 잃을 여유가 없는 거지.
전처리: 그러니까 주식하고 잃을 여유가 없는 거지.
--------------------------------------------------


In [ ]:
samples = [
    "",
    "   ",
    "[침묵]",
    "(무음)",
    "[기침] [잡음]",
    "(NO:)",
    "어...",
    "음...",
    "그 뭐냐 오늘은 목요일이여.",
    "[웃음] 잘 모르겠어요.",
    "<소음> 관리실에 전화해봐야제.",
]

for s in samples:
    cleaned = preprocess_stt_text(s)
    print("원본:", repr(s))
    print("전처리:", repr(cleaned))
    print("-" * 50)

원본: ''
전처리: ''
--------------------------------------------------
원본: '   '
전처리: ''
--------------------------------------------------
원본: '[침묵]'
전처리: ''
--------------------------------------------------
원본: '(무음)'
전처리: ''
--------------------------------------------------
원본: '[기침] [잡음]'
전처리: ''
--------------------------------------------------
원본: '(NO:)'
전처리: ''
--------------------------------------------------
원본: '어...'
전처리: '어...'
--------------------------------------------------
원본: '음...'
전처리: '음...'
--------------------------------------------------
원본: '그 뭐냐 오늘은 목요일이여.'
전처리: '그 뭐냐 오늘은 목요일이여.'
--------------------------------------------------
원본: '[웃음] 잘 모르겠어요.'
전처리: '잘 모르겠어요.'
--------------------------------------------------
원본: '<소음> 관리실에 전화해봐야제.'
전처리: '관리실에 전화해봐야제.'
--------------------------------------------------


In [ ]:
import re
from datetime import datetime


def preprocess_stt_text(text):
    """
    STT 결과를 분석용으로 최소 전처리한다.
    의미를 바꾸는 수정은 하지 않는다.
    """

    if text is None:
        return ""

    text = str(text).strip()

    # 라벨/비언어 표기 제거
    text = re.sub(r"\((NO:|웃음|기침|잡음|소음|침묵|무음|음악)\)", " ", text)
    text = re.sub(r"\[(NO:|웃음|기침|잡음|소음|침묵|무음|음악)\]", " ", text)
    text = re.sub(r"<(NO:|웃음|기침|잡음|소음|침묵|무음|음악)>", " ", text)

    # 괄호 없이 NO:가 들어오는 경우
    text = re.sub(r"\bNO:\s*", " ", text)

    # 줄바꿈, 탭, 여러 공백 정리
    text = re.sub(r"\s+", " ", text)

    # 양끝 따옴표 제거
    text = text.strip(" \"'“”‘’")

    return text.strip()


def calculate_response_time(question_provided_at, answer_started_at):
    """
    반응 시간 계산.
    question_provided_at: 문항 제공 완료 시각
    answer_started_at: 답변 시작 시각

    반환값: 초 단위 float
    """

    if question_provided_at is None or answer_started_at is None:
        return None

    # 문자열이면 datetime으로 변환
    if isinstance(question_provided_at, str):
        question_provided_at = datetime.fromisoformat(question_provided_at)

    if isinstance(answer_started_at, str):
        answer_started_at = datetime.fromisoformat(answer_started_at)

    response_time = (answer_started_at - question_provided_at).total_seconds()

    # 음수면 데이터 이상으로 보고 None 처리
    if response_time < 0:
        return None

    return round(response_time, 2)


def tokenize_korean_text(text):
    """
    공백 기준 어절 분리.
    문장부호는 제거하되, 발화 자체는 크게 바꾸지 않는다.
    """

    if text is None:
        return []

    text = str(text).strip()

    if text == "":
        return []

    # 문장부호를 공백으로 처리
    text = re.sub(r"[.,!?。！？…~]+", " ", text)

    # 괄호, 따옴표 등 분석 방해 기호 정리
    text = re.sub(r"[\"'“”‘’()\[\]{}<>]", " ", text)

    # 여러 공백 정리
    text = re.sub(r"\s+", " ", text).strip()

    if text == "":
        return []

    return text.split(" ")


def calculate_repetition_ratio(text):
    """
    반복어 비율 계산.
    기준: 같은 어절이 바로 연속해서 반복되는 경우를 반복으로 본다.

    예:
    '유월 유월 사일이에요' → 유월 1개 반복
    '어 어 그 뭐냐' → 어 1개 반복

    반환값: 백분율 float
    """

    tokens = tokenize_korean_text(text)

    if len(tokens) == 0:
        return 0.0

    repeated_count = 0

    for i in range(1, len(tokens)):
        if tokens[i] == tokens[i - 1]:
            repeated_count += 1

    ratio = repeated_count / len(tokens) * 100

    return round(ratio, 2)


def split_sentences(text):
    """
    문장 분리.
    STT 문장부호가 있으면 문장부호 기준으로 나누고,
    문장부호가 없으면 전체를 한 문장으로 본다.
    """

    if text is None:
        return []

    text = str(text).strip()

    if text == "":
        return []

    # 문장 종결 부호 기준 분리
    sentences = re.split(r"[.!?。！？]+", text)

    sentences = [s.strip() for s in sentences if s.strip()]

    # 문장부호가 없어도 발화가 있으면 한 문장
    if not sentences and text:
        sentences = [text]

    return sentences


def calculate_avg_sentence_length(text):
    """
    평균 문장 길이 계산.
    기준: 문장당 평균 어절 수

    반환값: float
    """

    if text is None:
        return 0.0

    text = str(text).strip()

    if text == "":
        return 0.0

    sentences = split_sentences(text)

    if len(sentences) == 0:
        return 0.0

    total_tokens = 0

    for sentence in sentences:
        tokens = tokenize_korean_text(sentence)
        total_tokens += len(tokens)

    avg_length = total_tokens / len(sentences)

    return round(avg_length, 2)


def analyze_speech_text(
    stt_text,
    question_provided_at=None,
    answer_started_at=None
):
    """
    STT 텍스트 하나에 대해 전처리 + 3개 지표 계산.
    """

    preprocessed_text = preprocess_stt_text(stt_text)

    response_time = calculate_response_time(
        question_provided_at,
        answer_started_at
    )

    repetition_ratio = calculate_repetition_ratio(preprocessed_text)

    avg_sentence_length = calculate_avg_sentence_length(preprocessed_text)

    return {
        "stt_text": stt_text,
        "preprocessed_text": preprocessed_text,
        "response_time": response_time,
        "repetition_ratio": repetition_ratio,
        "avg_sentence_length": avg_sentence_length,
    }

In [ ]:
samples = [
    "오늘은 유월 유월 사일 같아요.",
    "어 어 그 뭐냐 물 주문해브러.",
    "[잡음] 좋은 운동 방법 있으면은 저도 좀 부탁할게요.",
    "(NO:)차차차 뭐 이런 거 있잖아요 언니?",
    "",
]

for s in samples:
    result = analyze_speech_text(s)

    print("=" * 60)
    print("원본:", result["stt_text"])
    print("전처리:", result["preprocessed_text"])
    print("반복어 비율:", result["repetition_ratio"])
    print("평균 문장 길이:", result["avg_sentence_length"])

원본: 오늘은 유월 유월 사일 같아요.
전처리: 오늘은 유월 유월 사일 같아요.
반복어 비율: 20.0
평균 문장 길이: 5.0
원본: 어 어 그 뭐냐 물 주문해브러.
전처리: 어 어 그 뭐냐 물 주문해브러.
반복어 비율: 16.67
평균 문장 길이: 6.0
원본: [잡음] 좋은 운동 방법 있으면은 저도 좀 부탁할게요.
전처리: 좋은 운동 방법 있으면은 저도 좀 부탁할게요.
반복어 비율: 0.0
평균 문장 길이: 7.0
원본: (NO:)차차차 뭐 이런 거 있잖아요 언니?
전처리: 차차차 뭐 이런 거 있잖아요 언니?
반복어 비율: 0.0
평균 문장 길이: 6.0
원본: 
전처리: 
반복어 비율: 0.0
평균 문장 길이: 0.0
